## Article ↔ Document Group Analysis

Identifies the many-to-many relationship between articles and document groups: articles that appear in more than one reprint group.

- **Input:** `../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv`
- **Output:** `../_data/multi_group_articles.csv`

In [1]:
import pandas as pd

INFILE  = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "../_data/multi_group_articles.csv"

### Which articles belong to more than one document group?

In [2]:
df = pd.read_csv(INFILE, dtype=str).fillna("")

# Use compound_object rows only — one row per article per document group
# (excludes image child rows so each article-group pair is counted once)
compound = df[df["display_template"].str.lower().str.strip() == "compound_object"].copy()

article_groups = (
    compound.groupby("article_id")["group_reprint_id"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index()
    .rename(columns={"group_reprint_id": "document_groups"})
)
article_groups["num_groups"] = article_groups["document_groups"].apply(len)

multi = article_groups[article_groups["num_groups"] > 1].sort_values("num_groups", ascending=False)

print(f"Total articles: {len(article_groups)}")
print(f"Articles in multiple document groups: {len(multi)}\n")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
print(multi[["article_id", "num_groups", "document_groups"]].to_string(index=False))

Total articles: 481
Articles in multiple document groups: 13

            article_id  num_groups                                      document_groups
    CambriaFreeman1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
 SalemWeeklyReview1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
          Sheridan1925           2            [Sheridan1925_reprint, Terry1882_reprint]
   Sheridan1927_21(33)           2            [Sheridan1925_reprint, Terry1882_reprint]
   Sheridan1927_21(34)           2            [Sheridan1925_reprint, Terry1882_reprint]
ShipwreckedMariner1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
     Taylor1860_12(11)           2 [Russell1856_HCF_reprint, Taylor1860_12(11)_reprint]
     Taylor1860_13(16)           2 [Russell1856_HCF_reprint, Taylor1860_12(11)_reprint]
        TheGazette1879           2         [Stuart1878_reprint, TheGazette1879_reprint]
 TheNewfoundlander1879           2         [Stuart1878_rep

### Coverage analyses

The four tables below characterise the full set of 481 unique articles across key dimensions of the archive's data model.

In [3]:
# --- 1. Reprint group membership ---
standalone = compound[compound["group_reprint_id"].str.strip() == ""].drop_duplicates("article_id")
in_group   = compound[compound["group_reprint_id"].str.strip() != ""].drop_duplicates("article_id")

n_total     = compound["article_id"].nunique()
n_in_group  = in_group["article_id"].nunique()
n_standalone = standalone["article_id"].nunique()

print("=== 1. Reprint Group Membership ===\n")
print(f"{'Category':<40} {'Count':>5}  {'%':>6}")
print("-" * 54)
print(f"{'In at least one reprint group':<40} {n_in_group:>5}  {n_in_group/n_total*100:>5.1f}%")
print(f"{'Not in any reprint group (standalone)':<40} {n_standalone:>5}  {n_standalone/n_total*100:>5.1f}%")
print(f"{'Total unique articles':<40} {n_total:>5}")
print()
print("Standalone articles (publication, date):")
cols = ["article_id", "publication", "date"]
print(standalone[cols].sort_values("date").to_string(index=False))

=== 1. Reprint Group Membership ===

Category                                 Count       %
------------------------------------------------------
In at least one reprint group              362   75.3%
Not in any reprint group (standalone)      119   24.7%
Total unique articles                      481

Standalone articles (publication, date):
                     article_id                                                                                                                                publication       date
       NewhampshireSentinel1885                                                                                                                                                      
                   McCawley1996                                                                                 The First Angelinos: The Gabrielino Indians of Los Angeles           
                   Hardacre1879                                                                             

In [4]:
# --- 2. Source note coverage ---
sn = pd.read_csv("../_data/article_sourcenotes.csv")
sn_ids = set(sn["article_id"])

unique_articles = compound.drop_duplicates("article_id")[["article_id", "publication"]].copy()
unique_articles["has_source_note"] = unique_articles["article_id"].isin(sn_ids)

n_with_sn    = unique_articles["has_source_note"].sum()
n_without_sn = (~unique_articles["has_source_note"]).sum()

print("=== 2. Source Note Coverage ===\n")
print("Source notes describe the history and context of a periodical publication.")
print("They exist only for newspapers, magazines, and journals — not for books,")
print("manuscripts, theses, or government reports.\n")
print(f"{'Category':<45} {'Count':>5}  {'%':>6}")
print("-" * 58)
print(f"{'Articles with a source note (periodicals)':<45} {n_with_sn:>5}  {n_with_sn/n_total*100:>5.1f}%")
print(f"{'Articles without a source note (non-periodicals)':<45} {n_without_sn:>5}  {n_without_sn/n_total*100:>5.1f}%")
print(f"{'Total unique articles':<45} {n_total:>5}")
print()
print("Articles WITHOUT source notes (non-periodical sources):")
no_sn = unique_articles[~unique_articles["has_source_note"]].sort_values("publication")
print(no_sn[["article_id", "publication"]].to_string(index=False))

=== 2. Source Note Coverage ===

Source notes describe the history and context of a periodical publication.
They exist only for newspapers, magazines, and journals — not for books,
manuscripts, theses, or government reports.

Category                                      Count       %
----------------------------------------------------------
Articles with a source note (periodicals)       422   87.7%
Articles without a source note (non-periodicals)    59   12.3%
Total unique articles                           481

Articles WITHOUT source notes (non-periodical sources):
                 article_id                                                                                                                                publication
   NewhampshireSentinel1885                                                                                                                                           
           PointMuguHandout                                                              

In [5]:
# --- 3. Articles with document introductions ---
has_intro = compound[
    compound["document_intro"].str.strip() != ""
].drop_duplicates("article_id")[["article_id", "publication", "group_reprint_id", "document_intro"]].copy()

has_intro["intro_chars"] = has_intro["document_intro"].str.len()
has_intro["in_reprint_group"] = has_intro["group_reprint_id"].str.strip() != ""

print("=== 3. Articles with Document Introductions ===\n")
print(f"Articles with a document_intro: {len(has_intro)} of {n_total}\n")
print(f"{'article_id':<30} {'publication':<40} {'group_reprint_id':<35} {'chars':>6}  {'in group':>8}")
print("-" * 122)
for _, row in has_intro.sort_values("in_reprint_group", ascending=False).iterrows():
    grp = row["group_reprint_id"] if row["in_reprint_group"] else "(none — standalone)"
    print(f"{row['article_id']:<30} {row['publication']:<40} {grp:<35} {row['intro_chars']:>6}  {str(row['in_reprint_group']):>8}")
print()
print("Note: intros on group originals (Boston Atlas, Daily Alta, Hardacre/Scribner's)")
print("anchor three major reprint networks and tie directly to the literary tropes essay,")
print("which traces how language and ideas moved across these reprint chains.")
print("The Nidever and Dittman manuscript intros discuss the two key eyewitness sources")
print("whose accounts most shaped the historical record of the Lone Woman.")

=== 3. Articles with Document Introductions ===

Articles with a document_intro: 5 of 481

article_id                     publication                              group_reprint_id                     chars  in group
--------------------------------------------------------------------------------------------------------------------------
DailyAltaCalifornia1853        San Francisco Daily Alta California      DailyAltaCalifornia1853_reprint       1856      True
Hardacre1880_SM                Scribner's Monthly                       Hardacre1880_SM_reprint               4558      True
TheBostonAtlas1847             Boston Atlas                             TheBostonAtlas1847_reprint            2818      True
Dittman1878                    Manuscript C-D67                         (none — standalone)                   6186     False
Nidever1878                    Manuscript C-D133                        (none — standalone)                   5781     False

Note: intros on group originals (Bo

In [6]:
# --- 4. ObjectID counts: compound objects and image rows ---
total_rows     = len(df)
compound_rows  = len(df[df["display_template"].str.strip() == "compound_object"])
image_rows     = len(df[df["display_template"].str.strip() == "image"])
unique_oids    = df["objectid"].nunique()
unique_articles_n = compound["article_id"].nunique()  # 481

print("=== 4. ObjectID Counts ===\n")
print("Each row in the master CSV has a unique objectid.")
print("There are two row types:\n")
print(f"  compound_object rows (article parents):  {compound_rows}")
print(f"    = 481 unique articles")
print(f"    + 13 dual-group articles each counted twice")
print(f"    = 481 + 13 = 494\n")
print(f"  image rows (individual scan pages):      {image_rows}")
print(f"    Each compound_object parent has one image child per digitized page.\n")
print(f"  Total rows / unique objectids:           {total_rows}")
print()
avg_pages = image_rows / unique_articles_n
print(f"Average scan pages per article: {avg_pages:.1f}")
print()
print("Summary table:")
print(f"  {'Row type':<35} {'Count':>6}")
print(f"  {'-'*42}")
print(f"  {'compound_object (article parents)':<35} {compound_rows:>6}")
print(f"  {'image (digitized scan pages)':<35} {image_rows:>6}")
print(f"  {'Total objectids':<35} {total_rows:>6}")

=== 4. ObjectID Counts ===

Each row in the master CSV has a unique objectid.
There are two row types:

  compound_object rows (article parents):  494
    = 481 unique articles
    + 13 dual-group articles each counted twice
    = 481 + 13 = 494

  image rows (individual scan pages):      3934
    Each compound_object parent has one image child per digitized page.

  Total rows / unique objectids:           4428

Average scan pages per article: 8.2

Summary table:
  Row type                             Count
  ------------------------------------------
  compound_object (article parents)      494
  image (digitized scan pages)          3934
  Total objectids                       4428


### Export results to CSV

In [7]:
# Flatten document_groups list to semicolon-separated string for CSV storage
export = multi.copy()
export["document_groups"] = export["document_groups"].apply(lambda x: "; ".join(x))

export.to_csv(OUTFILE, index=False)
print(f"Saved {len(export)} rows to {OUTFILE}")

Saved 13 rows to ../_data/multi_group_articles.csv
